In [ ]:
import ipytest
import pytest

ipytest.autoconfig()

import numpy as np
import pandas as pd

try:
    import pymc as pm
except ImportError:
    pm = None

from preliz import predictive_explorer
from preliz.distributions import Gamma, Normal


In [ ]:
%%ipytest

from unittest.mock import patch

pytest.importorskip("ipywidgets")

x = np.linspace(0, 1, 100)


@pytest.fixture
def model():
    def a_preliz_model(a_mu, a_sigma, c_sigma=1):
        a = Normal(a_mu, a_sigma).rvs()
        c = Gamma(mu=2, sigma=c_sigma).rvs()
        a = np.exp(a)
        b = Normal(a * x, c).rvs()
        return b

    return a_preliz_model


def test_predictive_explorer(model):
    with patch("preliz.predictive.predictive_explorer.display") as mock_display:
        predictive_explorer(model, 10, "ecdf")

    mock_display.assert_called_once()
    widget = mock_display.call_args.args[0]
    descriptions = [
        child.description
        for child in widget.children[0].children
        if hasattr(child, "description")
    ]
    assert "a_mu (-inf, inf)" in descriptions
    assert "a_sigma (0, inf)" in descriptions
    assert "c_sigma (0, inf)" in descriptions


@pytest.mark.parametrize("kind_plot", ["hist", "kde", "ecdf"])
def test_predictive_explorer_kinds(model, kind_plot):
    with patch("preliz.predictive.predictive_explorer.display") as mock_display:
        predictive_explorer(model, 10, kind_plot)

    mock_display.assert_called_once()


def test_predictive_explorer_scalar_output():
    def scalar_model(mu):
        return Normal(mu, 1).rvs()

    with patch("preliz.predictive.predictive_explorer.display") as mock_display:
        predictive_explorer(scalar_model, 10, "ecdf")

    mock_display.assert_called_once()


def test_predictive_explorer_custom_plot(model):
    calls = []

    def lin_reg(predictions, ax):
        calls.append(predictions)
        ax.plot(x, predictions.T, "k.")

    with patch("preliz.predictive.predictive_explorer.display") as mock_display:
        predictive_explorer(model, 10, plot_func=lin_reg)

    mock_display.assert_called_once()
    assert calls


In [ ]:
%%ipytest

from unittest.mock import patch

pm = pytest.importorskip("pymc")


@pytest.fixture
def model():
    def a_pymc_model(a_mu, b_sigma=1):
        with pm.Model() as model:
            a = pm.Normal("a", a_mu, 1)
            b = pm.HalfNormal("b", b_sigma)
            c = pm.Normal("c", a, b, observed=[0] * 100)
        return model

    return a_pymc_model


@pytest.mark.parametrize("kind_plot", ["hist", "kde", "ecdf"])
def test_predictive_explorer(model, kind_plot):
    with patch("preliz.predictive.predictive_explorer.display") as mock_display:
        predictive_explorer(model, 10, kind_plot, engine="pymc")

    mock_display.assert_called_once()


def test_predictive_explorer_prior(model):
    with patch("preliz.predictive.predictive_explorer.display") as mock_display:
        predictive_explorer(model, 10, "ecdf", engine="pymc", group="prior")

    mock_display.assert_called_once()


In [ ]:
# %%ipytest

# rng = np.random.default_rng(1241)

# data = pd.DataFrame(
#     {
#         "y": rng.normal(size=117),
#         "x": rng.normal(size=117),
#     }
# )
# data.head()

# @pytest.fixture
# def model():
#     def a_bambi_model(a_mu, b_sigma=1):
#         prior = {"Intercept": bmb.Prior("Normal", mu=a_mu, sigma=b_sigma)}
#         a_model = bmb.Model("y ~ x", data, priors=prior)
#         return a_model
#     return a_bambi_model

# @pytest.mark.parametrize("iterations, kind_plot", [
#     (50, "hist"),
#     (10, "kde"),
#     (10, "ecdf"),
# ])
# def test_predictive_explorer(model, iterations, kind_plot):
#     predictive_explorer(model, iterations, kind_plot)